### This is the cookbook for our FalconOCR model.
This cookbook will show you how to setup the model,the best practices for using the model, and the various use cases.



### Setup

We use the **paged inference engine** (`OCRInferenceEngine`) which provides:
- Paged KV-cache for efficient memory usage
- Continuous batching with `torch.compile` + CUDA graph capture
- Both plain (full-page) and layout-based OCR modes


### Full page OCR

You should use this mode when the text is not too dense and you want low latency. In this mode, the layout model is not used.

In [ ]:
from IPython.display import display, Latex, HTML, Markdown


In [ ]:
import torch
from PIL import Image
from IPython.display import display, Latex, HTML, Markdown

from falcon_perception import OCR_MODEL_ID, cuda_timed, load_and_prepare_model, setup_torch_config
from falcon_perception.data import load_image, ImageProcessor
from falcon_perception.paged_inference import engine_config_for_gpu
from falcon_perception.paged_ocr_inference import OCRInferenceEngine

setup_torch_config()

model, tokenizer, model_args = load_and_prepare_model(
    hf_model_id=OCR_MODEL_ID,
    dtype="float32",
    compile=True,
)

image_processor = ImageProcessor(patch_size=16, merge_size=1)

cfg = engine_config_for_gpu(max_image_size=1024, dtype=model.dtype)
cfg.pop("max_hr_cache_entries", None)
cfg.pop("max_image_size", None)
print(f"Auto-config: {cfg}")

engine = OCRInferenceEngine(
    model, tokenizer, image_processor,
    max_seq_length=8192,
    capture_cudagraph=True,
    **cfg,
)

# Warmup absorbs torch.compile JIT + CUDA graph capture costs
print("Warmup run ...")
warmup_img = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/handwriting.jpg")
with cuda_timed(reset_peak_memory=False) as warmup_timer:
    engine.generate_plain([warmup_img], use_tqdm=False)
print(f"Warmup done in {warmup_timer.elapsed:.1f}s")

image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/handwriting.jpg")
texts = engine.generate_plain([image])
print(texts[0])

load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/handwriting.jpg")


### Element level OCR

To use element level OCR, you need to first run layout detection followed by OCR on the detected elements.
The layout detection model can be generic, however we use [PP-DocLayoutv3](https://huggingface.co/docs/transformers/main/model_doc/pp_doclayout_v3) for this task.
If you installed transformers>5 with pip, you already have access to the model.

You should use this mode when the text is dense and you want to get higher quality OCR.

In [ ]:
results = engine.generate_with_layout(
    [load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/real_news.jpg")],
)

for det in results[0]:
    display(Markdown(f"{det['text']}"))


The two stage pipeline depends critically on the layout model. If the layout model is not good, the OCR will not be good.
Hence, it might be useful to actually tune the layout model output based on your use case.
You visualize the layout and tune using the `score`. The layout label is `idx category score`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
from PIL import Image

def visualize_layout(image: Image.Image, detections: list[dict], max_dim: int = 1024):
    w_img, h_img = image.size
    scale = max_dim / max(w_img, h_img)
    fig_w, fig_h = w_img * scale / 100, h_img * scale / 100

    categories = sorted(set(d["category"] for d in detections))
    palette = list(mcolors.TABLEAU_COLORS.values())
    cat_to_color = {cat: palette[i % len(palette)] for i, cat in enumerate(categories)}

    fig, ax = plt.subplots(1, 1, figsize=(fig_w, fig_h))
    ax.imshow(image)

    for i, det in enumerate(detections):
        x0, y0, x1, y1 = det["bbox"]
        w, h = x1 - x0, y1 - y0
        color = cat_to_color[det["category"]]
        rgb = mcolors.to_rgb(color)

        rect = patches.Rectangle((x0, y0), w, h, linewidth=1.5, edgecolor=color, facecolor=(*rgb, 0.3))
        ax.add_patch(rect)
        label = f'{i}:{det["category"]}:{int(det["score"] * 100)}'
        ax.text(x0 - 2, y0 - 4, label, fontsize=9,
                color="white", backgroundcolor=color, fontweight="bold",
                va="bottom", ha="left")

    ax.set_axis_off()
    ax.set_title("Layout Detection", fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_layout(load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/real_news.jpg"), results[0])

### Use cases I - Handwriting

The model is trained on a diverse dataset of handwritten text, and can read legible handwriting with high accuracy. However, in our experience, certain forms of handwriting, especially stylistic cursive are very difficult to read, even for humans. Thus, the model outputs "plausible" text, which is not always correct.

In [ ]:
load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/hand_letter.jpg") 

In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/hand_letter.jpg")
texts = engine.generate_plain([image])
print(texts[0])


### Use Case II - Real World Images/Documents

The model can perform OCR even on images captured in the wild. 

In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/basma.jpg")
image.thumbnail((600, 600), Image.Resampling.LANCZOS)
image

In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/basma.jpg")
texts = engine.generate_plain([image])
print(texts[0])


In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/realworld_doc.jpg")
image.thumbnail((600, 600), Image.Resampling.LANCZOS)
image

In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/realworld_doc.jpg")
texts = engine.generate_plain([image])
display(Latex(texts[0]))


---
For this kind of document, you can use the `generate_with_layout` method, if the `generate` 
method struggles with the resolution.


In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/receipt.jpg")
texts = engine.generate_plain([image])
print(texts[0])

load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/receipt.jpg")


In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/walmart.jpg")
texts = engine.generate_plain([image])
print(texts[0])

load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/walmart.jpg")


### Use Case III - Formulas

The model is capable of outputting latex code for mathematical formulas, including alignment. 
However, note that there are many ways to represent the same formula, and the model may 
output different representations.

The default prompt for the model specifies `text`, we can generate latex code by specifying `formula` in the prompt as 

```python
texts = engine.generate_plain([image], category="formula")
```

In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/complex_formula.jpg")
texts = engine.generate_plain([image], category="formula")
display(Latex(texts[0]))

load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/complex_formula.jpg")


In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/simple_formula.jpg")
texts = engine.generate_plain([image], category="formula")
display(Latex(texts[0]))

load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/simple_formula.jpg")


### Use Case III - HTML tables

Similar to formulas, our model can output structured data in the form of HTML tables. The model handles complex layouts with cells spanning multiple rows and columns and/or empty cells.

The default prompt for the model specifies `text`, we can generate latex code by specifying `table` in the prompt as 

```python
texts = engine.generate_plain([image], category="table")
```

In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/simple_table.jpeg")
texts = engine.generate_plain([image], category="table")
display(HTML(texts[0]))

load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/simple_table.jpeg")


In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/table_complex.jpg")
texts = engine.generate_plain([image], category="table")
display(HTML(texts[0]))

load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/table_complex.jpg")


### Use Case IV - General OCR

For general OCR, we recommend using the `generate_with_layout` method. This will first run layout detection followed by OCR on the detected elements.

PP-DocLayoutv3 however has certain limitations, and in cases where the layout model is not able to detect the layout correctly, you can use the `generate` method.




In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/omnidocbench_283.jpg")
results = engine.generate_with_layout([image])
for det in results[0]:
    display(Markdown(f"{det['text']}"))

img = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/omnidocbench_283.jpg")
img.thumbnail((900, 900), Image.Resampling.LANCZOS)
img


##### Scanned Documents

The model also shows good performance on old/scanned documents, and is quite robust to the quality of the image. For this use case, the `generate` method provides robust performance. We show examples with both `generate` and `generate_with_layout` methods.

In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/scanned_1.jpg")
texts = engine.generate_plain([image])
print(texts[0])

img = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/scanned_1.jpg")
img.thumbnail((800, 800), Image.Resampling.LANCZOS)
img


In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/scanned_2.jpg")
texts = engine.generate_plain([image])
print(texts[0])

img = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/scanned_2.jpg")
img.thumbnail((800, 800), Image.Resampling.LANCZOS)
img


In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/scanned_3.jpg")
results = engine.generate_with_layout([image])
all_text = "\n".join([det["text"] for det in results[0]])
print(all_text)

img = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/scanned_3.jpg")
img.thumbnail((800, 800), Image.Resampling.LANCZOS)
img


In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/scanned_4.jpg")
results = engine.generate_with_layout([image])
all_text = "\n".join([det["text"] for det in results[0]])
print(all_text)

img = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/scanned_4.jpg")
img.thumbnail((800, 800), Image.Resampling.LANCZOS)
img


#### Scientific Papers

On scientific papers, the model can intermix tables, equations, and text. We recommend using the `generate_with_layout` method for this use case.

In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/3lm.jpg")
results = engine.generate_with_layout([image])
all_text = "\n".join([det["text"] for det in results[0]])
print(all_text)

img = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/3lm.jpg")
img.thumbnail((800, 800), Image.Resampling.LANCZOS)
img


In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/complex_paper.jpg")
results = engine.generate_with_layout([image])
all_text = "\n".join([det["text"] for det in results[0]])
display(Markdown(all_text))

img = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/complex_paper.jpg")
img.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
img


In [ ]:
image = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/table_paper.jpg")
results = engine.generate_with_layout([image])
all_text = "\n\n".join([det["text"] for det in results[0]])
display(Markdown(all_text))

img = load_image("https://huggingface.co/datasets/tiiuae/PBench/resolve/main/examples/table_paper.jpg")
img.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
img


In [ ]:
visualize_layout(image, results[0])